# We are going to make visuals for both MH and MK compounds

In [ ]:
import polars as pl
from pathlib import Path
import numpy as np
import pyboat
from pyboat import WAnalyzer
from pyboat import plotting
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import pandas as pd
from scipy.signal import savgol_filter

May (most recent) dataset

In [12]:
may_dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/enVision/enVision_pre"
                    "processed/2026-05-11._.PREPROCESSED.parquet")

may_detected_period = Path("/home/derfelt/PartchLabFiles/Circadian "
                       "Rhythms/Live Study/enVision/period_dete"
                       "ction_results/5-11_per-well_period_results.csv")

may_dataset = pl.scan_parquet(may_dataset_path).filter(pl.col("feature") == "CalcResultI")

may_tpoints = may_dataset.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_numpy()

may_detected_period = pl.read_csv(may_detected_period)
# display(detected_period)

may_platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

may_platemap = may_detected_period.join(may_platemap, on="Well", how='left')

mk_wells = (may_platemap
            .filter(pl.col("MoleculeID").str.contains("MK-", literal=True))
            .select("Well").to_series().to_list())

print(mk_wells)

print(len(may_tpoints))
display(may_dataset.head(2).collect())
display(may_platemap)

['F17', 'F18', 'F16', 'K16', 'K17', 'K18', 'F13', 'F14', 'F15', 'K13', 'K15', 'K14', 'F11', 'F12', 'F10', 'K10', 'K12', 'K11', 'F07', 'F09', 'F08', 'K07', 'K09', 'K08', 'F05', 'F06', 'F04', 'K04', 'K06', 'K05', 'G03', 'F22', 'G04', 'L04', 'L03', 'K22', 'F21', 'F20', 'F19', 'K20', 'K21', 'K19', 'E15', 'E17', 'E16', 'J16', 'J17', 'J15', 'E14', 'E13', 'E12', 'J13', 'J14', 'J12', 'E10', 'E09', 'E11', 'J10', 'J11', 'J09', 'E06', 'E07', 'E08', 'J07', 'J08', 'J06', 'E04', 'E05', 'E03', 'J04', 'J03', 'J05', 'E22', 'F03', 'E21', 'J21', 'J22', 'K03', 'E18', 'E19', 'E20', 'J18', 'J19', 'J20']
206


Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",30.0,"""CalcResultI""",15.122279,15.122279,null,15.17334,15.163128,15.14781,15.122279,15.096748,15.08143,15.071217
"""A01""",30.5,"""CalcResultI""",-4.79441,-4.79441,null,-4.740661,-4.751411,-4.767536,-4.79441,-4.821285,-4.837409,-4.848159


Well,group_label,detected_period,MoleculeID,strain,sample_type
str,str,f64,str,str,str
"""G06""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G07""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G05""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""L05""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
"""L06""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
…,…,…,…,…,…
"""O21""","""nan | nan""",25.153846,null,null,null
"""I03""","""nan | nan""",46.714286,null,null,null
"""N17""","""nan | nan""",23.357143,null,null,null


April Dataset (4-01)

In [16]:
april_dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                          "Study/enVision/enVision_preprocessed/30h-window._.20"
                          "26-04-01._.PREPROCESSED.parquet")

april_dataset = pl.scan_parquet(april_dataset_path).filter(pl.col("feature") == "CalcResultI")

april_detected_period = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/"
                             "Live Study/enVision/period_detection_results/4-"
                             "01_per-well_period_results.csv")
april_detected_period = pl.read_csv(april_detected_period)

april_platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                  "Study/platemaps/2026-04-01Echo"
                  "_transfer.xlsx", engine='openpyxl')
    .filter(pl.col("dex") == 'dex')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             ])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

april_platemap = april_detected_period.join(april_platemap, on="Well", how='left')

mh_wells = (april_platemap
            .filter(pl.col("MoleculeID").str.contains("MH-", literal=True))
            .select("Well")
            .to_series()
            .to_list())

display(april_platemap)
display(april_dataset.head(2).collect())
print(mh_wells)

Well,group_label,detected_period,MoleculeID,strain
str,str,f64,str,str
"""D17""","""0.1% DMSO | Bmal1""",33.571429,"""0.1% DMSO""","""Bmal1"""
"""D15""","""0.1% DMSO | Bmal1""",33.571429,"""0.1% DMSO""","""Bmal1"""
"""D05""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
"""G14""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
"""E05""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
…,…,…,…,…
"""K04""","""nan | nan""",26.111111,null,null
"""O04""","""nan | nan""",26.111111,null,null
"""K15""","""nan | nan""",26.111111,null,null


Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",30.0,"""CalcResultI""",-215.092717,-215.092717,null,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717
"""A01""",30.5,"""CalcResultI""",-124.756786,-124.756786,null,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786


['C08', 'D07', 'K17', 'C18', 'K12', 'M10', 'F11', 'M14', 'M07', 'D08', 'C10', 'K13', 'G12', 'H07', 'F17', 'H14', 'N20', 'J09', 'N10', 'K20', 'N03', 'E11', 'E07', 'K05', 'C06', 'G16', 'K22', 'E08', 'D12', 'I07', 'F21', 'D06', 'J07', 'F15', 'M13', 'I17']
